
---

## A. Main Assumptions of Linear Regression

For a linear regression model to provide reliable predictions and valid statistical inferences, it relies on several key assumptions. If these are violated, your model might be "biased" or "inefficient."

1. **Linearity**: The relationship between the independent variables ($X$) and the dependent variable ($y$) must be linear.
2. **Independence of Errors**: Observations should be independent of each other. This is crucial in time-series data to avoid **Autocorrelation**.
3. **Homoscedasticity**: The variance of residual terms (errors) should be constant across all levels of the independent variables. If the variance changes (e.g., spreads out like a fan), it’s called **Heteroscedasticity**.
4. **Normality of Residuals**: For any fixed value of $X$, the errors should be normally distributed. This is important for hypothesis testing and calculating confidence intervals.
5. **No Multicollinearity**: Independent variables should not be highly correlated with each other. High correlation makes it difficult to determine the individual effect of each predictor.

---

## B. R-squared vs. Adjusted R-squared

While both measure how well your model fits the data, they behave very differently when you start adding more variables.

| Feature | R-squared ($R^2$) | Adjusted R-squared |
| --- | --- | --- |
| **Definition** | The proportion of variance in the dependent variable explained by the model. | A modified version of $R^2$ that accounts for the number of predictors. |
| **Behavior** | **Always increases** (or stays the same) as you add more variables, even if they are useless "noise." | **Decreases** if the new variable does not improve the model's predictive power significantly. |
| **Reliability** | Can lead to overfitting by making a complex model look better than it is. | Provides a more honest assessment of model quality. |

---

## C. Regularization Techniques

Regularization prevents **overfitting** by adding a "penalty" term to the cost function, discouraging the model from relying too heavily on any single feature.

### 1. Ridge Regression (L2 Regularization)

Adds a penalty equal to the square of the magnitude of coefficients. It shrinks coefficients toward zero but never makes them exactly zero.
**Cost Function:**


$$J(\theta) = \sum_{i=1}^{n} (y_i - \hat{y}_i)^2 + \lambda \sum_{j=1}^{m} \theta_j^2$$

### 2. Lasso Regression (L1 Regularization)

Adds a penalty equal to the absolute value of the magnitude of coefficients. This can force some coefficients to be exactly zero, effectively performing **feature selection**.
**Cost Function:**


$$J(\theta) = \sum_{i=1}^{n} (y_i - \hat{y}_i)^2 + \lambda \sum_{j=1}^{m} |\theta_j|$$

### 3. Elastic Net

A hybrid approach that combines both L1 and L2 penalties. It’s useful when there are multiple correlated features.
**Cost Function:**


$$J(\theta) = \sum_{i=1}^{n} (y_i - \hat{y}_i)^2 + \lambda_1 \sum_{j=1}^{m} |\theta_j| + \lambda_2 \sum_{j=1}^{m} \theta_j^2$$

---

## D. Logistic Regression for Multiclass Classification

Standard Logistic Regression is a binary classifier (0 or 1). To handle multiple classes (e.g., Cat, Dog, Bird), we use two primary strategies:

### 1. One-vs-Rest (OvR) / One-vs-All

The model treats each class as a separate binary classification problem against all other classes.

* **Example**: Train Model 1 (Cat vs. [Dog, Bird]), Model 2 (Dog vs. [Cat, Bird]), etc.
* The class with the highest probability score is the final prediction.

### 2. Softmax Regression (Multinomial Logistic Regression)

Instead of multiple binary classifiers, it uses a single model with a **Softmax function** at the output layer. The Softmax function generalizes the sigmoid function to map the outputs into a probability distribution that sums to 1.


$$P(y=k | X) = \frac{e^{z_k}}{\sum_{j=1}^{K} e^{z_j}}$$

---

## E. Performance Metrics of Logistic Regression

Since Logistic Regression deals with classification, we don't use $R^2$. Instead, we use metrics derived from the **Confusion Matrix**.

```markdown
### 1. Accuracy
The ratio of correctly predicted observations to the total observations.
Formula: (TP + TN) / (TP + TN + FP + FN)

### 2. Precision (Positive Predictive Value)
Out of all predicted positives, how many were actually positive? 
Focuses on minimizing False Positives.
Formula: TP / (TP + FP)

### 3. Recall (Sensitivity)
Out of all actual positives, how many did we correctly identify? 
Focuses on minimizing False Negatives.
Formula: TP / (TP + FN)

### 4. F1-Score
The harmonic mean of Precision and Recall. It is the best metric 
when you have an imbalanced dataset.
Formula: 2 * (Precision * Recall) / (Precision + Recall)

### 5. ROC-AUC Score
- ROC (Receiver Operating Characteristic) curve plots the True Positive Rate 
  against the False Positive Rate.
- AUC (Area Under the Curve) measures the overall ability of the model 
  to distinguish between classes. 1.0 is perfect; 0.5 is no better than random guessing.

### 6. Log Loss (Cross-Entropy Loss)
The actual cost function used to train the model. It penalizes false 
classifications by looking at the probability confidence. 
A lower log loss indicates a better model.

```

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import ElasticNet, Lasso, LinearRegression, Ridge
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import cross_val_score, train_test_split

In [ ]:
mobile_data = pd.read_csv('Cellphone.csv')
mobile_data.head()

In [ ]:
mobile_data.info()

In [ ]:
mobile_data.describe().T


In [ ]:
mobile_data.isnull().sum()

In [ ]:
mobile_data.drop(columns=["Product_id","Sale"], inplace=True)
mobile_data.head()

In [ ]:
## Standardization
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
scaled_features = scaler.fit_transform(mobile_data.drop(columns=["Price"]))
mobile_data_scaled = pd.DataFrame(scaled_features, columns=mobile_data.columns[1:])
mobile_data_scaled["Price"] = mobile_data["Price"]
mobile_data_scaled
mobile_data_scaled.head()

In [ ]:
# First Model with Outliers
x1=mobile_data_scaled.drop(columns=["Price"])
y1=mobile_data_scaled["Price"]

In [ ]:
x_train,x_test,y_train,y_test = train_test_split(x1,y1,test_size=0.25,random_state=42)

In [ ]:
for i in mobile_data.columns:
    plt.figure(figsize=(8,4))
    sns.boxplot(mobile_data_scaled[i])
    plt.title(f'Boxplot of {i}')
    plt.show() 


In [ ]:
### Model Building : With Outliers
linear=LinearRegression()
linear.fit(x_train,y_train)

In [ ]:
## Try L1 Regularization and L2 Regularization and compare the results with Linear Regression. (ElasticNet, Lasso, Ridge) with Outliers
elastic_net = ElasticNet()
elastic_net.fit(x_train, y_train)
lasso = Lasso()
lasso.fit(x_train, y_train)
ridge = Ridge()
ridge.fit(x_train, y_train)
# Lasso Regression
lasso = Lasso(alpha=0.1)
lasso.fit(x_train, y_train)
print("***********"*7)
print("Lasso R2 Score (Train):", r2_score(y_train, lasso.predict(x_train)))
print("Lasso R2 Score (Test):", r2_score(y_test, lasso.predict(x_test)))

# Ridge Regression

ridge = Ridge(alpha=0.1)
ridge.fit(x_train, y_train)
print("***********"*7)
print("Ridge R2 Score (Train):", r2_score(y_train, ridge.predict(x_train)))
print("Ridge R2 Score (Test):", r2_score(y_test, ridge.predict(x_test)))

# ElasticNet Regression
elastic = ElasticNet(alpha=0.1, l1_ratio=0.5)

elastic.fit(x_train, y_train)
print("***********"*7)
print("ElasticNet R2 Score (Train):", r2_score(y_train, elastic.predict(x_train)))
print("ElasticNet R2 Score (Test):", r2_score(y_test, elastic.predict(x_test)))
print("***********"*7)


print("R2 Score (Train):", r2_score(y_train, linear.predict(x_train)))
print("R2 Score (Test):", r2_score(y_test, linear.predict(x_test)))
print("***********"*7)

In [ ]:
# Treating Outliers
Q1 = x1.quantile(0.25)
Q3 = x1.quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
x1_outliers_removed = x1[~((x1 < lower_bound) | (x1 > upper_bound)).any(axis=1)]
y1_outliers_removed = y1[x1_outliers_removed.index]
x_train_o,x_test_o,y_train_o,y_test_o = train_test_split(x1_outliers_removed,y1_outliers_removed,test_size=0.25,random_state=42)

In [ ]:
### Model Building : Without Outliers
linear_without_outliers=LinearRegression()
linear_without_outliers.fit(x_train_o,y_train_o)

In [ ]:
## Try L1 Regularization and L2 Regularization and compare the results with Linear Regression. (ElasticNet, Lasso, Ridge)


# Lasso Regression
lasso_without_outliers = Lasso(alpha=0.1)
lasso_without_outliers.fit(x_train_o, y_train_o)
print("Lasso R2 Score (Train):", r2_score(y_train_o, lasso_without_outliers.predict(x_train_o)))
print("Lasso R2 Score (Test):", r2_score(y_test_o, lasso_without_outliers.predict(x_test_o)))
print("***********"*7)

# Ridge Regression
ridge_without_outliers = Ridge(alpha=0.1)
ridge_without_outliers.fit(x_train_o, y_train_o)
print("Ridge R2 Score (Train):", r2_score(y_train_o, ridge_without_outliers.predict(x_train_o)))
print("Ridge R2 Score (Test):", r2_score(y_test_o, ridge_without_outliers.predict(x_test_o)))
print("***********"*7)


# ElasticNet Regression
elastic_without_outliers = ElasticNet(alpha=0.1, l1_ratio=0.5)
elastic_without_outliers.fit(x_train_o, y_train_o)
print("ElasticNet R2 Score (Train):", r2_score(y_train_o, elastic_without_outliers.predict(x_train_o)))
print("ElasticNet R2 Score (Test):", r2_score(y_test_o, elastic_without_outliers.predict(x_test_o)))
print("***********"*7)

print("R2 Score (Train):", r2_score(y_train_o, linear_without_outliers.predict(x_train_o)))
print("R2 Score (Test):", r2_score(y_test_o, linear_without_outliers.predict(x_test_o)))
print("***********"*7)



In [ ]:
import joblib
temp_scaler = joblib.load('scaler.pkl')
print(type(temp_scaler))

In [ ]:
joblib.dump(scaler, 'scaler.pkl')

https://mobilepredictionsanujsalwan.streamlit.app